# Giorno 3 — Image/Video Restoration & Caso Studio: Targa

**Obiettivi della giornata:**
1. Restoration di frame CCTV con Real-ESRGAN (super-resolution, denoising)
2. Ripetibilità: logging, salvataggio parametri, hash delle immagini
3. **Caso studio completo**: pipeline di lettura automatica di targhe degradate
   - Detection targa (YOLOv11 pre-addestrato su targhe)
   - Restoration (Real-ESRGAN)
   - OCR (EasyOCR) — lettura prima e dopo restoration

---
**Dataset:**
- Video CCTV a bassa risoluzione (`CCTV Action Recognition`)
- Immagini di targhe (`CCPD` — Chinese Car Plate Detection)

**Modelli:** Real-ESRGAN (tutto-in-uno per restoration), YOLOv11 (license plate), EasyOCR

> ℹ️ Anche oggi: solo inference, nessun retraining.

## 0. Setup

In [ ]:
!pip install -q basicsr facexlib gfpgan
!pip install -q realesrgan
!pip install -q ultralytics easyocr huggingface_hub
!pip install -q opencv-python-headless matplotlib tqdm pandas

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from IPython.display import HTML
from base64 import b64encode
import hashlib
import json
import datetime
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../utils')
from cv_utils import (mostra_frame, mostra_confronto, mostra_griglia,
                       stampa_info_video, estrai_frame, degrada_immagine, psnr)

def mostra_video_colab(path, width=700):
    with open(path, 'rb') as f:
        video_b64 = b64encode(f.read()).decode()
    return HTML(f'<video width="{width}" controls><source src="data:video/mp4;base64,{video_b64}" type="video/mp4"></video>')

print('✅ Import completati')

In [ ]:
# Configurazione percorsi
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/corso_cv/data/day_3')

DATA_DIR   = Path('../data/day_3')
PETS_DIR   = DATA_DIR / 'PETS2009'           # multi-view surveillance
ACTION_DIR = DATA_DIR / 'Videos' / 'Videos'  # CCTV Action Recognition (13 categorie)
CCPD_DIR   = DATA_DIR / 'CCPD2019'           # immagini di targhe

# Video multi-view PETS2009
pets_videos = sorted(PETS_DIR.glob('*.mp4'))
# Categorie azione
action_cats = sorted([d for d in ACTION_DIR.iterdir() if d.is_dir()])
# Immagini CCPD
image_files = sorted(CCPD_DIR.glob('**/*.jpg'))

print(f'PETS2009 multi-view: {len(pets_videos)} video')
for v in pets_videos:
    print(f'  {v.name}')

print(f'\nCCTV Action Recognition — {len(action_cats)} categorie:')
for cat in action_cats:
    n = len(list(cat.glob('*.mp4')))
    print(f'  {cat.name}: {n} video')

print(f'\nImmagini CCPD: {len(image_files)}')
from collections import Counter
cnt = Counter(f.parent.name for f in image_files)
for folder, n in sorted(cnt.items()):
    print(f'  {folder}: {n} immagini')

In [ ]:
# ── Preview: un frame per ogni categoria di azione ────────────────────────────
frames_action = []
titoli_action = []

for cat_dir in action_cats:
    vids = sorted(cat_dir.glob('*.mp4'))
    if vids:
        frame = estrai_frame(str(vids[0]), n=1)
        if frame:
            frames_action.append(frame[0])
            titoli_action.append(cat_dir.name)

if frames_action:
    mostra_griglia(frames_action, titoli_action, colonne=4)
    print(f'{len(frames_action)} categorie — clip reali da telecamere di sorveglianza')

---
## 1. Real-ESRGAN: Super-Resolution e Restoration

**Real-ESRGAN** è un modello di image restoration basato su GAN (Generative Adversarial Network), addestrato su degradazioni *sintetiche ma realistiche*.

A differenza dei modelli classici di super-resolution, Real-ESRGAN:
- Gestisce contemporaneamente: **super-resolution**, **denoising**, **deblur**, **de-compression**
- È addestrato su immagini CCTV, video compressi, immagini a bassa risoluzione
- Produce output fino a **4x** la risoluzione originale

**Varianti disponibili:**
- `RealESRGAN_x4plus`: uso generale, 4x upscale
- `RealESRGAN_x2plus`: 2x upscale, più veloce
- `RealESRGAN_x4plus_anime_6B`: ottimizzato per contenuti anime/cartoon
- `realesr-general-x4v3`: modello più leggero per degradazioni generali

Peso dei modelli: **~65MB** (scaricati automaticamente dal repo ufficiale)

In [ ]:
import torch
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan.archs.srvgg_arch import SRVGGNetCompact

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# ── Modello 1: RealESRGAN x4 (uso generale, alta qualità) ────────────────────
model_arch = RRDBNet(
    num_in_ch=3, num_out_ch=3, num_feat=64,
    num_block=23, num_grow_ch=32, scale=4
)

upsampler_x4 = RealESRGANer(
    scale=4,
    model_path='https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
    dni_weight=None,
    model=model_arch,
    tile=0,           # 0 = elaborazione senza tiling (più veloce, richiede più VRAM)
    tile_pad=10,
    pre_pad=0,
    half=True if device=='cuda' else False,  # fp16 su GPU
    device=device
)

print('✅ Real-ESRGAN x4 caricato')

In [ ]:
# ── Modello 2: RealESRGAN x2 (più veloce) ────────────────────────────────────
model_arch_x2 = RRDBNet(
    num_in_ch=3, num_out_ch=3, num_feat=64,
    num_block=23, num_grow_ch=32, scale=2
)

upsampler_x2 = RealESRGANer(
    scale=2,
    model_path='https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth',
    model=model_arch_x2,
    tile=0,
    tile_pad=10,
    pre_pad=0,
    half=True if device=='cuda' else False,
    device=device
)

print('✅ Real-ESRGAN x2 caricato')

In [ ]:
def esegui_realesrgan(img_bgr: np.ndarray, upsampler,
                       outscale: float = None) -> np.ndarray:
    """
    Applica Real-ESRGAN a un'immagine BGR.
    Restituisce l'immagine restored in BGR.
    """
    # Real-ESRGAN lavora in RGB
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    output, _ = upsampler.enhance(img_rgb, outscale=outscale)
    return cv2.cvtColor(output, cv2.COLOR_RGB2BGR)

---
## 2. Restoration di frame CCTV

In [ ]:
# Selezioniamo un video di sorveglianza con azione anomala
# Il dataset CCTV Action Recognition contiene clip reali da telecamere di sicurezza.
# Usiamo la categoria "fall" (caduta): tipica anomalia nei sistemi di sorveglianza.

action_fall = sorted((ACTION_DIR / 'fall').glob('*.mp4'))
VIDEO_CCTV  = str(action_fall[0]) if action_fall else None

if VIDEO_CCTV:
    print(f'Video selezionato: fall/{Path(VIDEO_CCTV).name}')
    stampa_info_video(VIDEO_CCTV)
    frame_cctv = estrai_frame(VIDEO_CCTV, n=1)[0]
    mostra_frame(frame_cctv, 'Frame CCTV — categoria: fall')
else:
    print('Nessun video trovato — creazione frame sintetico per demo')
    frame_hd   = np.random.randint(50, 200, (480, 640, 3), dtype=np.uint8)
    frame_cctv = degrada_immagine(frame_hd, scala=0.25, blur=3, rumore=20)

In [ ]:
# ── 2.1 Super-resolution x4 su frame CCTV ────────────────────────────────────
print('Elaborazione Real-ESRGAN x4... (può richiedere qualche secondo su CPU)')

frame_restored_x4 = esegui_realesrgan(frame_cctv, upsampler_x4)

print(f'Originale:  {frame_cctv.shape[1]}x{frame_cctv.shape[0]} px')
print(f'Restored x4: {frame_restored_x4.shape[1]}x{frame_restored_x4.shape[0]} px')

# Confronto
mostra_confronto(frame_cctv, frame_restored_x4,
                 f'CCTV originale ({frame_cctv.shape[1]}x{frame_cctv.shape[0]})',
                 f'Real-ESRGAN x4 ({frame_restored_x4.shape[1]}x{frame_restored_x4.shape[0]})')

In [ ]:
# ── 2.2 Confronto x2 vs x4 ───────────────────────────────────────────────────
frame_restored_x2 = esegui_realesrgan(frame_cctv, upsampler_x2)

# Portiamo tutto alla stessa risoluzione per confronto visivo
H_x4, W_x4 = frame_restored_x4.shape[:2]
frame_orig_up = cv2.resize(frame_cctv, (W_x4, H_x4), interpolation=cv2.INTER_NEAREST)
frame_x2_up   = cv2.resize(frame_restored_x2, (W_x4, H_x4), interpolation=cv2.INTER_LANCZOS4)

mostra_griglia(
    [frame_orig_up, frame_x2_up, frame_restored_x4],
    ['Originale (upsampled bicubico)', 'Real-ESRGAN x2', 'Real-ESRGAN x4'],
    colonne=3
)

### 2.5 SSIM — Structural Similarity Index

Il **PSNR** misura l'errore quadratico medio, ma non sempre riflette la percezione umana.  
Lo **SSIM** valuta tre componenti locali:
- **Luminanza** — livello medio di grigio
- **Contrasto** — deviazione standard locale
- **Struttura** — correlazione dei pattern (bordi, texture)

**Intervalli di qualità:** SSIM > 0.9 → ottima | 0.7–0.9 → buona | < 0.7 → degradata

**Vantaggio rispetto al PSNR:** due immagini con lo stesso PSNR ma diversa struttura (es. blur diffuso vs rumore casuale) hanno SSIM molto diversi. La mappa SSIM mostra esattamente *dove* si concentrano le differenze strutturali.

### 2.7 Multi-view: stessa scena da angolature diverse (PETS2009)

**PETS2009** (Performance Evaluation of Tracking and Surveillance) include più telecamere che riprendono la **stessa area** da angolature diverse.

Questo scenario è comune nella video-sorveglianza reale:
- angolazioni diverse → diverse occlusioni → diversa qualità delle riprese
- La restoration si comporta diversamente in base alla qualità originale del frame

Analizziamo come cambiano PSNR e SSIM dopo restoration al variare dell'angolazione.

In [ ]:
# ── 2.3 Restoration su immagine artificialmente degradata (PSNR misurabile) ──
# Per misurare il PSNR abbiamo bisogno dell'immagine originale (ground truth)
# Usiamo un frame HD e lo degradiamo noi stessi

# Prendiamo un frame dal video ad alta risoluzione se disponibile
# altrimenti usiamo il frame CCTV ridimensionato come proxy
frame_gt = cv2.resize(frame_cctv, 
                       (frame_cctv.shape[1]*4, frame_cctv.shape[0]*4),
                       interpolation=cv2.INTER_LANCZOS4)
frame_degradato = degrada_immagine(frame_gt, scala=0.25, blur=5, rumore=15)

print('Applicazione Real-ESRGAN sul frame degradato sinteticamente...')
frame_restored_sintetico = esegui_realesrgan(frame_degradato, upsampler_x4)

# Riportiamo tutte le immagini alla stessa risoluzione per il confronto
H_gt, W_gt = frame_gt.shape[:2]
frame_deg_up  = cv2.resize(frame_degradato, (W_gt, H_gt), interpolation=cv2.INTER_NEAREST)
frame_rest_up = cv2.resize(frame_restored_sintetico, (W_gt, H_gt), interpolation=cv2.INTER_LANCZOS4)

psnr_originale = psnr(frame_gt, frame_deg_up)
psnr_restored  = psnr(frame_gt, frame_rest_up)

print(f'\nPSNR originale vs degradato: {psnr_originale:.2f} dB')
print(f'PSNR originale vs restored:  {psnr_restored:.2f} dB')
print(f'Miglioramento PSNR:          {psnr_restored - psnr_originale:+.2f} dB')

mostra_griglia(
    [frame_gt, frame_deg_up, frame_rest_up],
    [f'Ground truth', f'Degradato (PSNR={psnr_originale:.1f}dB)', f'Restored (PSNR={psnr_restored:.1f}dB)'],
    colonne=3
)

In [ ]:
# ── 2.4b SSIM — Structural Similarity Index ───────────────────────────────────
try:
    from skimage.metrics import structural_similarity as ssim_fn
except ImportError:
    import subprocess; subprocess.run(['pip', 'install', '-q', 'scikit-image'])
    from skimage.metrics import structural_similarity as ssim_fn

def calcola_ssim(img_ref: np.ndarray, img_test: np.ndarray) -> float:
    """Calcola SSIM in scala di grigi; ridimensiona img_test se necessario."""
    if img_ref.shape != img_test.shape:
        img_test = cv2.resize(img_test, (img_ref.shape[1], img_ref.shape[0]),
                              interpolation=cv2.INTER_LANCZOS4)
    ref_g = cv2.cvtColor(img_ref,  cv2.COLOR_BGR2GRAY)
    tst_g = cv2.cvtColor(img_test, cv2.COLOR_BGR2GRAY)
    return ssim_fn(ref_g, tst_g, data_range=255)

# Utilizziamo le immagini già calcolate nella cella precedente:
#   frame_gt       → ground truth alta risoluzione
#   frame_deg_up   → degradato + upsampled alla risoluzione GT
#   frame_rest_up  → restored ESRGAN + upsampled

print(f'{"Scenario":<25} {"PSNR (dB)":>12} {"SSIM":>8}')
print('-' * 48)
for nome, img in [('Degradato', frame_deg_up), ('Restored (ESRGAN)', frame_rest_up)]:
    p = psnr(frame_gt, img)
    s = calcola_ssim(frame_gt, img)
    print(f'{nome:<25} {p:>12.2f} {s:>8.4f}')
print('\nℹ️  SSIM > 0.9 → ottima  |  0.7–0.9 → buona  |  < 0.7 → degradata')

# Mappa SSIM: dove si concentrano le differenze strutturali?
ref_g  = cv2.cvtColor(frame_gt,      cv2.COLOR_BGR2GRAY)
deg_g  = cv2.cvtColor(frame_deg_up,  cv2.COLOR_BGR2GRAY)
rest_g = cv2.cvtColor(frame_rest_up, cv2.COLOR_BGR2GRAY)

_, ssim_map_deg  = ssim_fn(ref_g, deg_g,  data_range=255, full=True)
_, ssim_map_rest = ssim_fn(ref_g, rest_g, data_range=255, full=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(ref_g, cmap='gray')
axes[0].set_title('Ground truth'); axes[0].axis('off')
axes[1].imshow(ssim_map_deg, cmap='RdYlGn', vmin=0, vmax=1)
axes[1].set_title(f'Mappa SSIM — Degradato\nSSIM={calcola_ssim(frame_gt, frame_deg_up):.4f}')
axes[1].axis('off')
axes[2].imshow(ssim_map_rest, cmap='RdYlGn', vmin=0, vmax=1)
axes[2].set_title(f'Mappa SSIM — Restored\nSSIM={calcola_ssim(frame_gt, frame_rest_up):.4f}')
axes[2].axis('off')
plt.suptitle('Mappa SSIM: verde = alta similarità, rosso = bassa similarità', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.8 Multi-view PETS2009: confronto angolazioni ────────────────────────────
print(f'PETS2009 — {len(pets_videos)} angolazioni della stessa scena\n')

frames_pets = []
titoli_pets = []
for v in pets_videos:
    f = estrai_frame(str(v), n=1)
    if f:
        frames_pets.append(f[0])
        titoli_pets.append(v.stem.replace('PETS2009_', ''))

mostra_griglia(frames_pets, titoli_pets, colonne=4)

# Restoration + metriche su 2 angolazioni (prima e ultima view)
print('\nRestoration x2 — confronto metriche su View_001 e View_005:')
views_test = [pets_videos[0], pets_videos[min(4, len(pets_videos)-1)]]

for v in views_test:
    frame      = estrai_frame(str(v), n=1)[0]
    frame_deg  = degrada_immagine(frame, scala=0.5, blur=5, rumore=15)
    frame_rest = esegui_realesrgan(frame_deg, upsampler_x2)

    H_r, W_r    = frame_rest.shape[:2]
    frame_gt_v  = cv2.resize(frame,     (W_r, H_r), interpolation=cv2.INTER_LANCZOS4)
    frame_deg_v = cv2.resize(frame_deg, (W_r, H_r), interpolation=cv2.INTER_NEAREST)

    p_deg  = psnr(frame_gt_v, frame_deg_v)
    p_rest = psnr(frame_gt_v, frame_rest)
    s_deg  = calcola_ssim(frame_gt_v, frame_deg_v)
    s_rest = calcola_ssim(frame_gt_v, frame_rest)

    print(f'\n  {v.stem}:')
    print(f'    Degradato → PSNR={p_deg:.1f}dB, SSIM={s_deg:.4f}')
    print(f'    Restored  → PSNR={p_rest:.1f}dB, SSIM={s_rest:.4f}  (Δ={p_rest-p_deg:+.1f}dB)')
    mostra_confronto(frame_deg_v, frame_rest,
                     f'{v.stem} — degradato (PSNR={p_deg:.1f}dB)',
                     f'{v.stem} — restored  (PSNR={p_rest:.1f}dB)')

In [ ]:
# ── 2.4 Restoration su sequenza video ────────────────────────────────────────
# Processiamo i primi N frame del video CCTV
N_FRAME_REST = 30  # pochi frame: Real-ESRGAN è lento su CPU
OUTPUT_REST  = '/tmp/cctv_restored.mp4'

if VIDEO_CCTV:
    cap = cv2.VideoCapture(VIDEO_CCTV)
    fps_r = cap.get(cv2.CAP_PROP_FPS)
    w_in  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h_in  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Il video di output ha risoluzione 4x
    writer = cv2.VideoWriter(OUTPUT_REST, cv2.VideoWriter_fourcc(*'mp4v'),
                             fps_r, (w_in * 4, h_in * 4))

    for i in tqdm(range(N_FRAME_REST), desc='Video restoration'):
        ret, frame = cap.read()
        if not ret:
            break
        restored = esegui_realesrgan(frame, upsampler_x4)
        writer.write(restored)

    cap.release()
    writer.release()
    print(f'\nVideo restored salvato: {OUTPUT_REST}')
    mostra_video_colab(OUTPUT_REST)
else:
    print('Nessun video disponibile — saltiamo la sezione')

---
## 3. Ripetibilità: logging, parametri e hash

In un contesto professionale, la **ripetibilità** è fondamentale:
- Devo poter ricostruire esattamente come è stata prodotta un'immagine/output
- Devo poter verificare che un'immagine non sia stata modificata dopo il processing
- Devo tracciare quale versione del modello ha prodotto quale output

**Strumenti che usiamo:**
- **Hash SHA-256**: impronta digitale unica dell'immagine
- **JSON log**: registro di ogni operazione eseguita
- **Parametri salvati**: configurazione completa riproducibile

In [ ]:
def hash_immagine(img: np.ndarray) -> str:
    """Calcola l'hash SHA-256 dei pixel dell'immagine."""
    return hashlib.sha256(img.tobytes()).hexdigest()


class SessionLogger:
    """
    Logger per tracciare operazioni di processing su immagini/video.
    Salva un file JSON con tutti i dettagli della sessione.
    """

    def __init__(self, output_path: str):
        self.output_path = output_path
        self.log = {
            'sessione': {
                'data': datetime.datetime.now().isoformat(),
                'device': device,
            },
            'operazioni': []
        }

    def registra(self, operazione: str, input_info: dict,
                  output_info: dict, parametri: dict):
        """Registra un'operazione nel log."""
        entry = {
            'timestamp': datetime.datetime.now().isoformat(),
            'operazione': operazione,
            'input': input_info,
            'output': output_info,
            'parametri': parametri,
        }
        self.log['operazioni'].append(entry)
        self._salva()

    def _salva(self):
        with open(self.output_path, 'w') as f:
            json.dump(self.log, f, indent=2, ensure_ascii=False)

    def stampa_riepilogo(self):
        print(f'Log sessione: {self.output_path}')
        print(f'Operazioni registrate: {len(self.log["operazioni"])}')
        for op in self.log['operazioni']:
            print(f"  [{op['timestamp'][:19]}] {op['operazione']}")

In [ ]:
# ── 3.1 Esempio di pipeline con logging ──────────────────────────────────────
LOG_PATH = '/tmp/sessione_restoration.json'
logger = SessionLogger(LOG_PATH)

# Parametri della sessione
PARAMS_RESTORATION = {
    'modello': 'RealESRGAN_x4plus',
    'scale': 4,
    'tile': 0,
    'half_precision': device == 'cuda',
    'device': device,
}

# --- Operazione 1: restoration di un frame ---
frame_input = frame_cctv.copy()
hash_input = hash_immagine(frame_input)

frame_output = esegui_realesrgan(frame_input, upsampler_x4)
hash_output = hash_immagine(frame_output)

logger.registra(
    operazione='real_esrgan_restoration',
    input_info={
        'risoluzione': f'{frame_input.shape[1]}x{frame_input.shape[0]}',
        'sha256': hash_input,
    },
    output_info={
        'risoluzione': f'{frame_output.shape[1]}x{frame_output.shape[0]}',
        'sha256': hash_output,
    },
    parametri=PARAMS_RESTORATION
)

print(f'Hash input:  {hash_input[:16]}...')
print(f'Hash output: {hash_output[:16]}...')
print('Gli hash sono diversi ✅ (l\'immagine è stata modificata, come atteso)')

In [ ]:
# ── 3.2 Verifica integrità: stessa immagine → stesso hash ────────────────────
frame_copia = frame_output.copy()
hash_copia = hash_immagine(frame_copia)

print(f'Hash output originale: {hash_output[:32]}...')
print(f'Hash copia:            {hash_copia[:32]}...')
print(f'Identici: {hash_output == hash_copia} ✅')

# Modifica minima → hash completamente diverso
frame_modificato = frame_output.copy()
frame_modificato[0, 0, 0] += 1  # cambia un solo pixel
hash_modificato = hash_immagine(frame_modificato)

print(f'\nHash dopo modifica 1px: {hash_modificato[:32]}...')
print(f'Diverso dall\'originale: {hash_output != hash_modificato} ✅')

In [ ]:
# ── 3.3 Salvataggio e caricamento parametri ───────────────────────────────────
PARAMS_FILE = '/tmp/params_restoration.json'

# Salvataggio
with open(PARAMS_FILE, 'w') as f:
    json.dump(PARAMS_RESTORATION, f, indent=2)
print(f'Parametri salvati in: {PARAMS_FILE}')

# Caricamento e riuso
with open(PARAMS_FILE, 'r') as f:
    params_caricati = json.load(f)

print('\nParametri caricati:')
for k, v in params_caricati.items():
    print(f'  {k}: {v}')

logger.stampa_riepilogo()

In [ ]:
# Visualizzazione del log JSON
with open(LOG_PATH) as f:
    print(f.read())

### 4.2b Ground Truth incorporato nel filename CCPD

Una caratteristica unica di **CCPD** è che il **ground truth è codificato nel filename** stesso — non serve un file di annotazione separato.

Il formato è:
```
{area}-{tiltV}_{tiltH}-{bbox_x1}&{y1}_{x2}&{y2}-{corners}-{prov}_{city}_{c1}_{c2}_{c3}_{c4}_{c5}-{brightness}-{blur}.jpg
```

Questo ci permette di:
- Confrontare direttamente il testo OCR con la targa reale
- Selezionare immagini per livello di sfocatura o angolo di ripresa
- Valutare quantitativamente il miglioramento portato dalla restoration

In [ ]:
# ── 4.2b Decoder filename CCPD + visualizzazione GT ──────────────────────────
CCPD_PROVINCES = ['皖','沪','津','渝','冀','豫','云','辽','黑','湘',
                   '皖','鲁','新','苏','浙','赣','鄂','桂','甘','晋',
                   '蒙','陕','吉','闽','贵','粤','川','青','琼','宁','琼']
CCPD_ALPHABET  = list('ABCDEFGHJKLMNPQRSTUVWXYZ')          # senza I e O
CCPD_ADS       = list('ABCDEFGHJKLMNPQRSTUVWXYZ0123456789') # senza I e O

def decodifica_ccpd(filename: str) -> dict:
    """
    Decodifica bbox, targa GT, angolo, luminosità e sfocatura
    direttamente dal filename CCPD.
    Formato: area-tiltV_tiltH-bbox-corners-plate_indices-brightness-blur.jpg
    """
    stem = Path(filename).stem
    parts = stem.split('-')
    if len(parts) < 7:
        return {'errore': 'formato non riconosciuto'}
    try:
        tilt_v, tilt_h = map(int, parts[1].split('_'))
        bbox_pts       = parts[2].split('_')
        x1, y1 = map(int, bbox_pts[0].split('&'))
        x2, y2 = map(int, bbox_pts[1].split('&'))
        char_idxs = list(map(int, parts[4].split('_')))
        provincia = CCPD_PROVINCES[char_idxs[0]] if char_idxs[0] < len(CCPD_PROVINCES) else '?'
        citta     = CCPD_ALPHABET[char_idxs[1]]  if char_idxs[1] < len(CCPD_ALPHABET)  else '?'
        resto     = ''.join(CCPD_ADS[i] if i < len(CCPD_ADS) else '?' for i in char_idxs[2:])
        return {
            'bbox':        (x1, y1, x2, y2),
            'angolo':      (tilt_v, tilt_h),
            'targa_gt':    f'{provincia}{citta}·{resto}',
            'luminosita':  int(parts[5]),
            'sfocatura':   int(parts[6]),
        }
    except (IndexError, ValueError) as e:
        return {'errore': str(e)}

# ── Test: decodifica e visualizzazione bbox GT ────────────────────────────────
print('=== Ground Truth codificato nel filename CCPD ===\n')
frames_ccpd_gt = []
titoli_ccpd_gt = []

for img_path in image_files[:6]:
    info = decodifica_ccpd(img_path.name)
    if 'errore' in info:
        continue
    print(f'📁 {img_path.parent.name} / {img_path.name[:45]}')
    print(f'   Targa GT:   {info["targa_gt"]}')
    print(f'   BBox:       {info["bbox"]}')
    print(f'   Angolo:     V={info["angolo"][0]}° H={info["angolo"][1]}°')
    print(f'   Luminosità: {info["luminosita"]}  |  Sfocatura: {info["sfocatura"]}\n')

    img = cv2.imread(str(img_path))
    if img is None:
        continue
    x1, y1, x2, y2 = info['bbox']
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
    cv2.putText(img, info['targa_gt'], (x1, max(y1-10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    frames_ccpd_gt.append(img)
    titoli_ccpd_gt.append(f'{img_path.parent.name}\nGT: {info["targa_gt"]}')

if frames_ccpd_gt:
    mostra_griglia(frames_ccpd_gt, titoli_ccpd_gt, colonne=3)

---
## 4. Caso Studio: License Plate Restoration & OCR

Mettiamo insieme tutto quello che abbiamo imparato in una pipeline realistica:

**Il problema:** un sistema di sorveglianza riceve video a bassa qualità. Le targhe dei veicoli sono illeggibili. Vogliamo:
1. Rilevare automaticamente le targhe nei frame
2. Applicare restoration per aumentarne la leggibilità
3. Leggere automaticamente il testo (OCR)
4. Confrontare la lettura prima e dopo restoration

**Pipeline:**
```
Frame CCTV → [YOLO targa] → crop targa → [Real-ESRGAN] → [EasyOCR] → testo
                                ↓
                         [EasyOCR su originale]
```

In [ ]:
# ── 4.1 Caricamento YOLO per license plate detection ─────────────────────────
# Modello YOLOv11 pre-addestrato per riconoscimento targhe
# Fonte: https://huggingface.co/morsetechlab/yolov11-license-plate-detection

from huggingface_hub import hf_hub_download
from ultralytics import YOLO

# Download dei pesi da HuggingFace (~14MB)
model_path = hf_hub_download(
    repo_id='morsetechlab/yolov11-license-plate-detection',
    filename='license_plate_detector.pt'
)

model_plate = YOLO(model_path)
print(f'Modello targhe caricato: {model_path}')
print(f'Classi: {model_plate.names}')

In [ ]:
# ── 4.2 Caricamento EasyOCR ───────────────────────────────────────────────────
import easyocr

# EasyOCR supporta decine di lingue; per targhe europee usiamo 'en'
# Il download dei modelli OCR avviene al primo utilizzo (~100MB)
reader_ocr = easyocr.Reader(['en'], gpu=(device=='cuda'), verbose=False)
print('✅ EasyOCR caricato')

### 4.10 Pipeline batch su tutto il subset CCPD

Eseguiamo la pipeline **detection → crop → OCR** su tutte le **135 immagini** (9 categorie × 15 immagini) e confrontiamo il tasso di successo dell'OCR per categoria:

| Categoria | Descrizione |
|---|---|
| `ccpd_base` | Condizioni normali (baseline) |
| `ccpd_blur` | Sfocatura intenzionale |
| `ccpd_db` | Bassa luminosità |
| `ccpd_challenge` | Casi difficili misti |
| `ccpd_rotate` | Targa ruotata |
| `ccpd_tilt` | Angolo di ripresa inclinato |
| `ccpd_weather` | Condizioni meteo avverse |
| `ccpd_fn` / `ccpd_np` | Falsi negativi / No plate |

Questo mostra dove la restoration con Real-ESRGAN porterebbe il maggiore beneficio.

In [ ]:
def rileva_targhe(img_bgr: np.ndarray, conf_th: float = 0.3) -> list:
    """
    Rileva le targhe nell'immagine.
    Restituisce lista di (x1, y1, x2, y2, conf).
    """
    ris = model_plate(img_bgr, verbose=False, conf=conf_th)
    boxes = []
    for box in ris[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        boxes.append((x1, y1, x2, y2, conf))
    return boxes


def leggi_targa(img_bgr: np.ndarray, soglia_conf: float = 0.3) -> str:
    """
    Esegue OCR sull'immagine (o crop di una targa).
    Restituisce il testo con confidenza più alta.
    """
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    risultati = reader_ocr.readtext(img_rgb)
    if not risultati:
        return '(nessun testo rilevato)'
    # Prende il testo con confidenza più alta
    best = max(risultati, key=lambda x: x[2])
    testo, conf = best[1], best[2]
    if conf < soglia_conf:
        return f'(bassa confidenza: {testo})'
    return testo.upper()


def estrai_crop_targa(img: np.ndarray, x1, y1, x2, y2,
                        padding: int = 5) -> np.ndarray:
    """Estrae il crop della targa con padding."""
    H, W = img.shape[:2]
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(W, x2 + padding)
    y2 = min(H, y2 + padding)
    return img[y1:y2, x1:x2]

In [ ]:
# ── 4.10 Pipeline batch: detection + OCR su tutto il subset CCPD ─────────────
# 9 sottocartelle × 15 immagini = 135 immagini totali
# ⏳ Qualche minuto (detection + OCR senza restoration)

import pandas as pd
risultati_batch = []

for cat_dir in sorted(CCPD_DIR.iterdir()):
    if not cat_dir.is_dir() or cat_dir.name == 'splits':
        continue
    imgs = sorted(cat_dir.glob('*.jpg'))
    if not imgs:
        continue

    n_det_ok = 0
    n_ocr_ok = 0

    for img_path in tqdm(imgs, desc=f'{cat_dir.name:22s}', leave=False):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        info_gt = decodifica_ccpd(img_path.name)
        targhe  = rileva_targhe(img, conf_th=0.25)
        if not targhe:
            continue
        n_det_ok += 1
        x1, y1, x2, y2, _ = max(targhe, key=lambda t: t[4])
        crop  = estrai_crop_targa(img, x1, y1, x2, y2)
        testo = leggi_targa(crop)
        ocr_ok = '(' not in testo
        if ocr_ok:
            n_ocr_ok += 1
        risultati_batch.append({
            'categoria': cat_dir.name,
            'gt':        info_gt.get('targa_gt', '?'),
            'ocr':       testo,
            'ocr_ok':    ocr_ok,
        })

    print(f'{cat_dir.name:24s}: det {n_det_ok}/{len(imgs)}, OCR ok {n_ocr_ok}/{max(n_det_ok,1)}')

df_batch = pd.DataFrame(risultati_batch)

# Riepilogo per categoria
riepilogo_batch = df_batch.groupby('categoria').agg(
    n_campioni   = ('gt', 'count'),
    ocr_successi = ('ocr_ok', 'sum'),
).assign(tasso_ocr=lambda x: (x['ocr_successi'] / x['n_campioni'] * 100).round(1))

print('\n=== Riepilogo batch ===')
print(riepilogo_batch.to_string())

fig, ax = plt.subplots(figsize=(12, 5))
riepilogo_batch['tasso_ocr'].plot(kind='bar', ax=ax, color='steelblue', alpha=0.85)
ax.set_ylabel('Tasso OCR successo (%)')
ax.set_title('EasyOCR success rate per categoria CCPD\n(detection → crop → OCR senza restoration)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.set_ylim(0, 110)
for p in ax.patches:
    ax.text(p.get_x() + p.get_width()/2, p.get_height() + 1,
            f'{p.get_height():.0f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()
print('\n💡 Applicare Real-ESRGAN prima dell\'OCR dovrebbe aumentare il tasso,')
print('   specialmente per "blur" e "db" (sfocatura e bassa luminosità).')

In [ ]:
# ── 4.3 Test su immagini CCPD ─────────────────────────────────────────────────
# Carichiamo le immagini di targhe disponibili
if image_files:
    img_plate_path = str(image_files[0])
    img_plate = cv2.imread(img_plate_path)
    print(f'Immagine targa: {image_files[0].name}')
    print(f'Risoluzione: {img_plate.shape[1]}x{img_plate.shape[0]}')
    mostra_frame(img_plate, 'Immagine originale CCPD')
else:
    # Creiamo un'immagine di test sintetica
    print('Nessuna immagine trovata — uso frame video come fallback')
    img_plate = estrai_frame(VIDEO_CCTV, n=1)[0] if VIDEO_CCTV else frame_cctv

In [ ]:
# ── 4.4 Pipeline completa: detection → crop → restoration → OCR ──────────────

def pipeline_targa(img_bgr: np.ndarray, logger: SessionLogger = None,
                    scala_degrado: float = 1.0) -> dict:
    """
    Pipeline completa per lettura targhe.
    
    Args:
        img_bgr: immagine di input (frame video)
        logger: istanza SessionLogger per il logging
        scala_degrado: se < 1.0, degrada artificialmente l'immagine
    
    Returns:
        dict con risultati per ogni targa rilevata
    """
    risultati_pipeline = []

    # Step 0: degradazione opzionale
    if scala_degrado < 1.0:
        img_proc = degrada_immagine(img_bgr, scala=scala_degrado, blur=3, rumore=10)
    else:
        img_proc = img_bgr.copy()

    # Step 1: detection targhe
    targhe = rileva_targhe(img_proc, conf_th=0.25)
    print(f'  Targhe rilevate: {len(targhe)}')

    for i, (x1, y1, x2, y2, conf_det) in enumerate(targhe):
        print(f'\n  --- Targa {i+1} (conf={conf_det:.2f}) ---')

        # Step 2: crop della targa
        crop_orig = estrai_crop_targa(img_proc, x1, y1, x2, y2)
        if crop_orig.size == 0:
            continue

        # Step 3: OCR sull'originale (o degradato)
        testo_orig = leggi_targa(crop_orig)
        print(f'  OCR originale:  "{testo_orig}"')

        # Step 4: restoration con Real-ESRGAN
        crop_restored = esegui_realesrgan(crop_orig, upsampler_x4)

        # Step 5: OCR sull'immagine restored
        testo_restored = leggi_targa(crop_restored)
        print(f'  OCR restored:   "{testo_restored}"')

        # PSNR (se non c'è degrado, non ha senso calcolarlo)
        psnr_val = None
        if scala_degrado < 1.0:
            crop_gt = estrai_crop_targa(img_bgr, x1, y1, x2, y2)
            crop_gt_up = cv2.resize(crop_gt, (crop_restored.shape[1], crop_restored.shape[0]),
                                     interpolation=cv2.INTER_LANCZOS4)
            psnr_val = psnr(crop_gt_up, crop_restored)
            print(f'  PSNR migliorato: {psnr_val:.2f} dB')

        res = {
            'bbox': (x1, y1, x2, y2),
            'conf_detection': conf_det,
            'crop_originale': crop_orig,
            'crop_restored': crop_restored,
            'ocr_originale': testo_orig,
            'ocr_restored': testo_restored,
            'psnr': psnr_val,
            'hash_orig': hash_immagine(crop_orig),
            'hash_rest': hash_immagine(crop_restored),
        }
        risultati_pipeline.append(res)

        if logger:
            logger.registra(
                operazione='license_plate_pipeline',
                input_info={'bbox': list(map(int, [x1,y1,x2,y2])),
                             'sha256_crop': res['hash_orig']},
                output_info={'sha256_restored': res['hash_rest'],
                              'psnr_db': psnr_val},
                parametri={'modello_detection': 'yolov11-license-plate',
                            'modello_restoration': 'RealESRGAN_x4plus',
                            'ocr_engine': 'EasyOCR',
                            'scala_degrado': scala_degrado}
            )

    return risultati_pipeline


print('Pipeline definita ✅')

In [ ]:
# ── 4.5 Esecuzione pipeline sull'immagine CCPD ───────────────────────────────
print('=== Pipeline License Plate Restoration ===')
print('Input: immagine originale (nessun degrado artificiale)\n')

ris_pipeline = pipeline_targa(img_plate, logger=logger, scala_degrado=1.0)

In [ ]:
# ── 4.6 Visualizzazione risultati ─────────────────────────────────────────────
if ris_pipeline:
    for i, res in enumerate(ris_pipeline):
        print(f'\n=== Targa {i+1} ===')
        print(f'  OCR originale: {res["ocr_originale"]}')
        print(f'  OCR restored:  {res["ocr_restored"]}')

        mostra_confronto(
            res['crop_originale'],
            res['crop_restored'],
            f'Originale\nOCR: "{res["ocr_originale"]}"',
            f'Real-ESRGAN x4\nOCR: "{res["ocr_restored"]}"'
        )
else:
    print('Nessuna targa rilevata nell\'immagine')
    print('Prova con un\'altra immagine da data/day_3')

In [ ]:
# ── 4.7 Test con degrado artificiale ─────────────────────────────────────────
# Degrado l'immagine per simulare una telecamera di bassa qualità
print('=== Pipeline su immagine DEGRADATA (simula CCTV bassa qualità) ===')
print('Degrado: scala 50%, blur lieve, rumore σ=10\n')

ris_degradato = pipeline_targa(img_plate, logger=logger, scala_degrado=0.5)

In [ ]:
# ── 4.8 Visualizzazione comparativa completa ──────────────────────────────────
if ris_pipeline and ris_degradato:
    # Confronto: originale puro vs degradato+restored vs originale+restored
    crop_orig_clean = ris_pipeline[0]['crop_originale']
    crop_rest_clean = ris_pipeline[0]['crop_restored']
    crop_orig_deg   = ris_degradato[0]['crop_originale'] if ris_degradato else None
    crop_rest_deg   = ris_degradato[0]['crop_restored'] if ris_degradato else None

    titoli = [
        f'Originale\n"{ris_pipeline[0]["ocr_originale"]}"',
        f'Restored (da orig.)\n"{ris_pipeline[0]["ocr_restored"]}"',
    ]
    frame_list = [crop_orig_clean, crop_rest_clean]

    if crop_orig_deg is not None:
        titoli += [
            f'Degradato\n"{ris_degradato[0]["ocr_originale"]}"',
            f'Restored (da deg.)\n"{ris_degradato[0]["ocr_restored"]}"',
        ]
        frame_list += [crop_orig_deg, crop_rest_deg]

    mostra_griglia(frame_list, titoli, colonne=2)
    print('\nRiepilogo log sessione:')
    logger.stampa_riepilogo()

In [ ]:
# ── 4.9 (Opzionale) Pipeline su frame video ───────────────────────────────────
# Uncommentare per eseguire su un frame del video CCTV

# if VIDEO_CCTV:
#     frame_video = estrai_frame(VIDEO_CCTV, n=1)[0]
#     print('=== Pipeline su frame video CCTV ===')
#     ris_video = pipeline_targa(frame_video, logger=logger, scala_degrado=1.0)
#
#     # Visualizzazione del frame con bounding box
#     frame_ann = frame_video.copy()
#     for res in ris_video:
#         x1, y1, x2, y2 = res['bbox']
#         cv2.rectangle(frame_ann, (x1, y1), (x2, y2), (0, 255, 0), 2)
#         cv2.putText(frame_ann, res['ocr_restored'],
#                     (x1, y1-8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
#     mostra_frame(frame_ann, 'Risultati pipeline su frame video')

---
## 5. Riepilogo del corso

### Giorno 3 — Cosa abbiamo fatto:
1. **Restoration** di frame CCTV con Real-ESRGAN (super-resolution 4x)
2. **Ripetibilità**: logging JSON, hash SHA-256, salvataggio parametri
3. **Caso studio completo**: detection targa → restoration → OCR

---

### Riepilogo delle 3 giornate

| | Giorno 1 | Giorno 2 | Giorno 3 |
|---|---|---|---|
| **Topic** | Detection & Tracking | Counting & Traiettorie | Restoration & OCR |
| **Librerie** | YOLO, BoxMOT, OpenCV | supervision, norfair | Real-ESRGAN, EasyOCR |
| **Output** | Bounding box + ID | Conteggi + heatmap | Immagini restaurate + testo |
| **Metrica chiave** | Confidenza, ID switch | N° attraversamenti, densità | PSNR, leggibilità OCR |

### Concetti trasversali applicati:
- **Pre-trained models**: nessun training, solo inference
- **Pipeline composizione**: combinare modelli diversi per task complessi
- **Ripetibilità**: logging, hash, parametri salvati
- **Trade-off**: qualità vs velocità, soglie di confidenza, degrado vs restoration

### 🔍 Domande finali:
- Quali elementi di questo corso potete applicare subito al vostro lavoro?
- Dove vedereste i maggiori problemi in produzione?
- Come garantireste l'affidabilità di una pipeline come quella del caso studio in un contesto legale/forense?

In [ ]:
# ── ESERCIZIO FINALE ─────────────────────────────────────────────────────────
# Scegliete una delle seguenti sfide:
#
# A) Provate la pipeline di restoration su più immagini dalla cartella day_3
#    e create un report con PSNR e risultati OCR
#
# B) Combinate il tracking del Giorno 1 con la restoration del Giorno 3:
#    applicare Real-ESRGAN ai crop dei veicoli tracciati in data/day_1
#
# C) Aggiungete la verifica dell'integrità all'output del Giorno 2:
#    salvare le heatmap con hash e log dei parametri

print('Buon lavoro! 🎉')